# DPO Training with Unsloth

This notebook performs preference optimization using Direct Preference Optimization (DPO).

Given structured (prompt, chosen, rejected) pairs constructed in the previous stage, we optimize the policy model to increase the likelihood of preferred responses relative to rejected ones.

We use:

- Parameter-efficient fine-tuning (LoRA)
- 4-bit quantization (QLoRA-style efficiency)
- Unsloth acceleration for memory-efficient training
- TRL’s DPOTrainer for stable preference learning

This stage converts structured preference supervision into aligned model behavior.


In [ ]:
import os
import torch
from accelerate import Accelerator
from datasets import load_from_disk

from utils.dpo_training_helpers import (
    preprocess_dpo,
    load_unsloth_model,
    apply_lora,
    build_dpo_trainer,
)

In [ ]:
accelerator = Accelerator()
print("Using accelerator:", accelerator.state)
print("Num processes:", accelerator.num_processes)

In [ ]:
import os
from pathlib import Path
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

try:
    REPO_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    REPO_ROOT = Path.cwd()


TRAIN_PATH = REPO_ROOT / "Skydpo_dataset" / "train"
VAL_PATH   = REPO_ROOT / "Skydpo_dataset" / "validation"

OUTPUT_DIR = REPO_ROOT / "models" / "SkyDPO_Qwen27"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [ ]:
train_dataset = load_from_disk(TRAIN_PATH)
eval_dataset  = load_from_disk(VAL_PATH)

print("Train columns:", train_dataset.column_names)
print("Eval columns:", eval_dataset.column_names)


### Preparing Data for DPO

DPO requires structured triplets:

- ```prompt```
- ```chosen```
- ```rejected```

We extract the human prompt from conversation-style data and preserve relevant metadata for analysis.
This preprocessing ensures compatibility with the DPO training objective introduced by Rafailov et al. (2023).

In [ ]:
train_dataset = train_dataset.map(
    preprocess_dpo,
    remove_columns=train_dataset.column_names,
)

eval_dataset = eval_dataset.map(
    preprocess_dpo,
    remove_columns=eval_dataset.column_names,
)

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))
print("Sample:", train_dataset[0])


### Loading Quantized Base Model

We load the base instruction-tuned model using 4-bit quantization to reduce memory footprint.

This enables efficient fine-tuning on limited hardware while preserving performance, following the QLoRA paradigm.

Key configuration:

- 4-bit weight quantization
- Flash Attention enabled
- Controlled maximum sequence length

This setup balances computational efficiency and training stability.


### Parameter-Efficient Fine-Tuning (LoRA)

Rather than updating all model parameters, we apply Low-Rank Adaptation (LoRA).

LoRA injects trainable rank-decomposition matrices into attention and feed-forward projections, significantly reducing the number of trainable parameters while maintaining expressivity.

This allows scalable preference optimization without full model fine-tuning.

In [ ]:
model, tokenizer = load_unsloth_model(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
)

model = apply_lora(model)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

### Direct Preference Optimization (DPO)

We train the model using the DPO objective (Rafailov et al., 2023), which eliminates the need for an explicit reward model.

Given a prompt $x$, chosen response $y_w$ and rejected response $y_l$:

$$
m(x, y_w, y_l) =
\log \frac{\pi_\theta(y_w \mid x)}{\pi_\theta(y_l \mid x)}
-
\log \frac{\pi_{\text{ref}}(y_w \mid x)}{\pi_{\text{ref}}(y_l \mid x)}
$$

The **DPO loss** is:

$$
\mathcal{L}_{\text{DPO}}(\theta)
=
-\mathbb{E}_{(x,y_w,y_l)}
\left[
\log \sigma\left(\beta \cdot m(x,y_w,y_l)\right)
\right]
$$
where:
- πθ: trainable policy
- π_ref: frozen reference policy
- β: temperature parameter
- σ(·): sigmoid function


This formulation directly aligns the model with preference data while avoiding instability associated with RL-based optimization.

We use:

- Cosine learning rate scheduling
- Gradient accumulation for stability
- 8-bit AdamW optimizer
- Mixed precision training


### Key Training Choices

- β = 0.1: Controls preference margin strength; smaller values prevent overconfidence.
- Learning rate = 1.8e-6: Low LR stabilizes preference optimization.
- Gradient accumulation = 16: Enables effective large batch size without memory explosion.
- LoRA rank = 32: Balances adaptation capacity and parameter efficiency.
- These settings follow common best practices for preference-based fine-tuning.

In [ ]:
trainer = build_dpo_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    output_dir=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
)

In [ ]:
print("Starting DPO training on 2× L4 GPUs...")
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Training complete! Model saved to: {OUTPUT_DIR}")

### References

- Rafailov et al., 2023. Direct Preference Optimization: Your Language Model is Secretly a Reward Model. https://arxiv.org/abs/2305.18290
- Hu et al., 2021. LoRA: Low-Rank Adaptation of Large Language Models. https://arxiv.org/abs/2106.09685
- Dettmers et al., 2023. QLoRA: Efficient Finetuning of Quantized LLMs.https://arxiv.org/abs/2305.14314